# Performance Tuning and Optimization

This notebook provides practical guidance for optimizing cross-validation performance, managing memory, and speeding up training.

## What you'll learn:
- Memory optimization techniques
- Speed optimization strategies
- GPU utilization best practices
- Debugging slow CV runs
- Resource monitoring and management

## 1. Setup and Resource Assessment

In [ ]:
import pandas as pd
import numpy as np
import psutil
import torch
import time
import gc
import tracemalloc
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
import sys
sys.path.append('../..')

print("=" * 70)
print("SYSTEM RESOURCE ASSESSMENT")
print("=" * 70)

In [ ]:
# CPU Information
print("\nCPU Information:")
print(f"  Physical cores: {psutil.cpu_count(logical=False)}")
print(f"  Logical cores:  {psutil.cpu_count(logical=True)}")
print(f"  Current usage:  {psutil.cpu_percent(interval=1)}%")
print(f"  Frequency:      {psutil.cpu_freq().current:.0f} MHz")

# Memory Information
mem = psutil.virtual_memory()
print("\nMemory Information:")
print(f"  Total:     {mem.total / 1e9:.1f} GB")
print(f"  Available: {mem.available / 1e9:.1f} GB")
print(f"  Used:      {mem.used / 1e9:.1f} GB ({mem.percent}%)")

# GPU Information
print("\nGPU Information:")
if torch.cuda.is_available():
    print(f"  Device:    {torch.cuda.get_device_name(0)}")
    print(f"  Count:     {torch.cuda.device_count()}")
    
    props = torch.cuda.get_device_properties(0)
    print(f"  Memory:    {props.total_memory / 1e9:.1f} GB")
    print(f"  Compute:   {props.major}.{props.minor}")
    
    # Current GPU memory usage
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"  Allocated: {allocated:.2f} GB")
        print(f"  Reserved:  {reserved:.2f} GB")
else:
    print("  No GPU available - using CPU")

# Disk Information
disk = psutil.disk_usage('/')
print("\nDisk Information:")
print(f"  Total:     {disk.total / 1e9:.1f} GB")
print(f"  Available: {disk.free / 1e9:.1f} GB")
print(f"  Used:      {disk.percent}%")

## 2. Memory Optimization Strategies

In [ ]:
print("=" * 70)
print("MEMORY OPTIMIZATION TECHNIQUES")
print("=" * 70)

# Create sample data
n_samples = 10000
n_features = 100

# Technique 1: Use appropriate dtypes
print("\n1. Data Type Optimization:")

# Original (float64)
df_float64 = pd.DataFrame(np.random.randn(n_samples, n_features))
mem_float64 = df_float64.memory_usage(deep=True).sum() / 1e6

# Optimized (float32)
df_float32 = df_float64.astype('float32')
mem_float32 = df_float32.memory_usage(deep=True).sum() / 1e6

print(f"  Float64 memory: {mem_float64:.2f} MB")
print(f"  Float32 memory: {mem_float32:.2f} MB")
print(f"  Savings: {(1 - mem_float32/mem_float64)*100:.1f}%")

# Clean up
del df_float64, df_float32
gc.collect()

In [ ]:
# Technique 2: Chunked processing
print("\n2. Chunked Processing:")

def process_in_chunks(df, chunk_size=1000):
    """Process large DataFrame in chunks to manage memory."""
    n_chunks = len(df) // chunk_size + (1 if len(df) % chunk_size else 0)
    results = []
    
    for i in range(n_chunks):
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, len(df))
        
        # Process chunk
        chunk = df.iloc[start_idx:end_idx]
        result = chunk.mean()  # Example processing
        results.append(result)
        
        # Clear chunk from memory
        del chunk
        
    return pd.concat(results)

# Demonstrate memory efficiency
df_large = pd.DataFrame(np.random.randn(50000, 50))
print(f"  DataFrame size: {df_large.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"  Processing in chunks of 5000 rows...")

# Track memory
mem_before = psutil.Process().memory_info().rss / 1e6
result = process_in_chunks(df_large, chunk_size=5000)
mem_after = psutil.Process().memory_info().rss / 1e6

print(f"  Memory delta: {mem_after - mem_before:.1f} MB")

del df_large
gc.collect()

In [ ]:
# Technique 3: Selective column loading
print("\n3. Selective Column Loading:")

# Create sample parquet file
df_full = pd.DataFrame({
    'unique_id': 'BTC',
    'ds': pd.date_range('2024-01-01', periods=10000, freq='15min'),
    'y': np.random.randn(10000) * 0.01,
    **{f'feature_{i}': np.random.randn(10000) for i in range(50)}
})

# Save to parquet
temp_file = 'temp_data.parquet'
df_full.to_parquet(temp_file)

# Load all columns
mem_before = psutil.Process().memory_info().rss / 1e6
df_all = pd.read_parquet(temp_file)
mem_all = psutil.Process().memory_info().rss / 1e6 - mem_before

del df_all
gc.collect()

# Load selective columns
mem_before = psutil.Process().memory_info().rss / 1e6
df_selective = pd.read_parquet(
    temp_file, 
    columns=['unique_id', 'ds', 'y', 'feature_0', 'feature_1']
)
mem_selective = psutil.Process().memory_info().rss / 1e6 - mem_before

print(f"  All columns memory:       {mem_all:.1f} MB")
print(f"  Selective columns memory: {mem_selective:.1f} MB")
print(f"  Savings: {(1 - mem_selective/mem_all)*100:.1f}%")

# Clean up
import os
os.remove(temp_file)
del df_full, df_selective
gc.collect()

## 3. Speed Optimization Techniques

In [ ]:
print("=" * 70)
print("SPEED OPTIMIZATION STRATEGIES")
print("=" * 70)

# Technique 1: Reduce training iterations for pilot
print("\n1. Adaptive Training Configuration:")

def get_adaptive_config(phase='pilot', gpu_available=torch.cuda.is_available()):
    """Get optimized configuration based on phase and resources."""
    
    if phase == 'pilot':
        config = {
            'max_steps': 1000,      # Reduced from 20000
            'val_check_steps': 50,  # More frequent validation
            'batch_size': 128 if gpu_available else 32,
            'n_windows': 3,         # Fewer CV windows
            'early_stop_patience_steps': 100
        }
    elif phase == 'development':
        config = {
            'max_steps': 5000,
            'val_check_steps': 100,
            'batch_size': 256 if gpu_available else 64,
            'n_windows': 6,
            'early_stop_patience_steps': 200
        }
    else:  # production
        config = {
            'max_steps': 20000,
            'val_check_steps': 200,
            'batch_size': 512 if gpu_available else 128,
            'n_windows': 10,
            'early_stop_patience_steps': 400
        }
    
    return config

# Show configurations
for phase in ['pilot', 'development', 'production']:
    config = get_adaptive_config(phase)
    print(f"\n{phase.capitalize()} Configuration:")
    for key, value in config.items():
        print(f"  {key:25s}: {value}")

In [ ]:
# Technique 2: Parallel model training
print("\n2. Parallel Model Training:")

from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import multiprocessing as mp

def train_model_dummy(model_name, config):
    """Dummy training function for demonstration."""
    time.sleep(0.5)  # Simulate training time
    return f"{model_name} trained with config: {config['max_steps']} steps"

# Sequential execution
models = ['NHITS', 'TiDE', 'NBEATSx', 'PatchTST']
config = {'max_steps': 1000}

print("\nSequential execution:")
start = time.time()
for model in models:
    result = train_model_dummy(model, config)
sequential_time = time.time() - start
print(f"  Time: {sequential_time:.2f} seconds")

# Parallel execution
print("\nParallel execution:")
start = time.time()
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(train_model_dummy, model, config) for model in models]
    results = [f.result() for f in futures]
parallel_time = time.time() - start
print(f"  Time: {parallel_time:.2f} seconds")
print(f"  Speedup: {sequential_time/parallel_time:.1f}x")

In [ ]:
# Technique 3: Caching and memoization
print("\n3. Caching Strategy:")

from functools import lru_cache
import hashlib

class CVCache:
    """Simple cache for CV results."""
    
    def __init__(self, cache_dir='cache'):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)
    
    def _get_hash(self, model_name, config):
        """Generate hash for cache key."""
        key = f"{model_name}_{str(config)}"
        return hashlib.md5(key.encode()).hexdigest()
    
    def get(self, model_name, config):
        """Retrieve from cache if exists."""
        cache_file = self.cache_dir / f"{self._get_hash(model_name, config)}.parquet"
        if cache_file.exists():
            return pd.read_parquet(cache_file)
        return None
    
    def set(self, model_name, config, results):
        """Store results in cache."""
        cache_file = self.cache_dir / f"{self._get_hash(model_name, config)}.parquet"
        results.to_parquet(cache_file)

# Demonstrate caching
cache = CVCache()

# First run - no cache
model = 'NHITS'
config = {'max_steps': 1000, 'batch_size': 128}

start = time.time()
cached_result = cache.get(model, config)
if cached_result is None:
    # Simulate CV computation
    time.sleep(0.5)
    result = pd.DataFrame({'predictions': np.random.randn(100)})
    cache.set(model, config, result)
    print(f"  First run (computed): {time.time() - start:.3f} seconds")
else:
    result = cached_result
    print(f"  First run (cached): {time.time() - start:.3f} seconds")

# Second run - from cache
start = time.time()
cached_result = cache.get(model, config)
if cached_result is not None:
    print(f"  Second run (cached): {time.time() - start:.3f} seconds")

# Clean up
import shutil
shutil.rmtree('cache')

## 4. GPU Optimization

In [ ]:
print("=" * 70)
print("GPU OPTIMIZATION TECHNIQUES")
print("=" * 70)

if torch.cuda.is_available():
    # Technique 1: Mixed precision training
    print("\n1. Mixed Precision Training:")
    print("   Enables automatic mixed precision (AMP) for faster training")
    print("   Configuration:")
    print("     - Use torch.cuda.amp.autocast()")
    print("     - 30-50% speedup on modern GPUs")
    print("     - Minimal accuracy loss")
    
    # Technique 2: Optimal batch size
    print("\n2. Batch Size Optimization:")
    
    def find_optimal_batch_size(model_memory_mb=500):
        """Estimate optimal batch size based on GPU memory."""
        total_memory = torch.cuda.get_device_properties(0).total_memory / 1e6
        available_memory = total_memory * 0.8  # Leave 20% buffer
        
        # Rough estimate: batch_size = available_memory / model_memory
        optimal_batch = int(available_memory / model_memory_mb)
        
        # Round to power of 2 for efficiency
        optimal_batch = 2 ** int(np.log2(optimal_batch))
        
        return min(optimal_batch, 512)  # Cap at 512
    
    optimal = find_optimal_batch_size()
    print(f"   Recommended batch size: {optimal}")
    print(f"   Based on available GPU memory")
    
    # Technique 3: GPU memory management
    print("\n3. GPU Memory Management:")
    
    # Clear cache
    torch.cuda.empty_cache()
    print(f"   Cache cleared")
    
    # Set memory fraction
    torch.cuda.set_per_process_memory_fraction(0.8)
    print(f"   Memory fraction set to 80%")
    
    # Monitor memory
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"   Current allocated: {allocated:.2f} GB")
    print(f"   Current reserved:  {reserved:.2f} GB")
    
else:
    print("\nNo GPU available. CPU optimization tips:")
    print("  1. Use smaller batch sizes (32-64)")
    print("  2. Reduce model complexity")
    print("  3. Enable multi-threading with OMP_NUM_THREADS")
    print("  4. Consider cloud GPU instances for production")

## 5. Cross-Validation Performance Profiling

In [ ]:
print("=" * 70)
print("CV PERFORMANCE PROFILING")
print("=" * 70)

class CVProfiler:
    """Profile CV execution to identify bottlenecks."""
    
    def __init__(self):
        self.timings = {}
        self.memory = {}
    
    def profile_step(self, step_name, func, *args, **kwargs):
        """Profile a single CV step."""
        # Memory before
        mem_before = psutil.Process().memory_info().rss / 1e6
        
        # Time execution
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        
        # Memory after
        mem_after = psutil.Process().memory_info().rss / 1e6
        
        # Store metrics
        self.timings[step_name] = elapsed
        self.memory[step_name] = mem_after - mem_before
        
        return result
    
    def report(self):
        """Generate performance report."""
        print("\nTiming Report:")
        total_time = sum(self.timings.values())
        for step, time_val in sorted(self.timings.items(), key=lambda x: x[1], reverse=True):
            pct = time_val / total_time * 100
            print(f"  {step:20s}: {time_val:6.2f}s ({pct:5.1f}%)")
        
        print("\nMemory Report:")
        for step, mem in sorted(self.memory.items(), key=lambda x: x[1], reverse=True):
            print(f"  {step:20s}: {mem:6.1f} MB")

# Simulate CV profiling
profiler = CVProfiler()

# Simulate CV steps
def dummy_data_prep():
    time.sleep(0.2)
    return pd.DataFrame(np.random.randn(1000, 10))

def dummy_model_fit(df):
    time.sleep(0.5)
    return "model"

def dummy_cv_run(model):
    time.sleep(0.8)
    return pd.DataFrame(np.random.randn(500, 5))

def dummy_metrics(cv_results):
    time.sleep(0.3)
    return {"scrps": 0.42}

# Profile each step
df = profiler.profile_step("Data Preparation", dummy_data_prep)
model = profiler.profile_step("Model Fitting", dummy_model_fit, df)
cv_results = profiler.profile_step("CV Execution", dummy_cv_run, model)
metrics = profiler.profile_step("Metrics Computation", dummy_metrics, cv_results)

# Generate report
profiler.report()

print("\nBottleneck Analysis:")
slowest = max(profiler.timings.items(), key=lambda x: x[1])
print(f"  Slowest step: {slowest[0]} ({slowest[1]:.2f}s)")
print(f"  Focus optimization efforts here for maximum impact")

## 6. Debugging Slow CV Runs

In [ ]:
print("=" * 70)
print("DEBUGGING SLOW CV RUNS")
print("=" * 70)

def diagnose_cv_performance(n_samples, n_features, n_models, n_windows, batch_size):
    """Diagnose potential performance issues."""
    
    issues = []
    recommendations = []
    
    # Check data size
    data_size_mb = n_samples * n_features * 8 / 1e6  # float64
    if data_size_mb > 1000:
        issues.append(f"Large dataset: {data_size_mb:.0f} MB")
        recommendations.append("- Use float32 instead of float64")
        recommendations.append("- Consider feature selection")
    
    # Check CV complexity
    total_fits = n_models * n_windows
    if total_fits > 20:
        issues.append(f"Many model fits: {total_fits}")
        recommendations.append("- Reduce n_windows for pilot")
        recommendations.append("- Run models in parallel")
    
    # Check batch size
    if torch.cuda.is_available():
        if batch_size < 128:
            issues.append(f"Small batch size: {batch_size}")
            recommendations.append("- Increase batch_size for GPU")
    else:
        if batch_size > 64:
            issues.append(f"Large batch size for CPU: {batch_size}")
            recommendations.append("- Reduce batch_size for CPU")
    
    # Memory estimate
    est_memory_gb = (data_size_mb * total_fits) / 1000
    available_gb = psutil.virtual_memory().available / 1e9
    if est_memory_gb > available_gb * 0.8:
        issues.append(f"Memory pressure: need {est_memory_gb:.1f} GB, have {available_gb:.1f} GB")
        recommendations.append("- Process models sequentially")
        recommendations.append("- Use chunked CV windows")
    
    return issues, recommendations

# Example diagnosis
issues, recs = diagnose_cv_performance(
    n_samples=50000,
    n_features=256,
    n_models=6,
    n_windows=10,
    batch_size=512
)

print("\nIdentified Issues:")
for issue in issues:
    print(f"  ⚠️  {issue}")

print("\nRecommendations:")
for rec in recs:
    print(f"  {rec}")

## 7. Optimization Checklist

In [ ]:
print("=" * 70)
print("CV OPTIMIZATION CHECKLIST")
print("=" * 70)

checklist = """
✅ BEFORE RUNNING CV:

1. Data Optimization
   □ Convert to float32 if precision allows
   □ Remove unnecessary columns
   □ Use parquet format for I/O
   □ Ensure data is sorted by timestamp

2. Configuration Tuning
   □ Use pilot config for initial runs
   □ Set appropriate batch_size for hardware
   □ Enable early stopping
   □ Reduce max_steps for testing

3. Resource Management
   □ Check available memory
   □ Clear GPU cache if using CUDA
   □ Close unnecessary applications
   □ Monitor system resources

✅ DURING CV:

4. Monitoring
   □ Watch memory usage
   □ Check GPU utilization
   □ Monitor training loss convergence
   □ Track time per window

5. Quick Fixes
   □ Reduce batch_size if OOM
   □ Enable gradient checkpointing
   □ Use mixed precision training
   □ Process models sequentially

✅ AFTER CV:

6. Analysis
   □ Profile slowest steps
   □ Check for memory leaks
   □ Analyze model convergence
   □ Document optimal settings

7. Cleanup
   □ Clear cache directories
   □ Remove temporary files
   □ Free GPU memory
   □ Save only necessary artifacts
"""

print(checklist)

## 8. Configuration Templates

In [ ]:
print("=" * 70)
print("OPTIMIZED CONFIGURATION TEMPLATES")
print("=" * 70)

import yaml

# Memory-constrained configuration
memory_config = {
    'name': 'Memory Optimized',
    'description': 'For systems with <16GB RAM',
    'models': {
        'batch_size': 32,
        'max_steps': 5000,
        'gradient_clip_val': 1.0,
        'accumulate_grad_batches': 4  # Simulate larger batch
    },
    'cv': {
        'n_windows': 3,
        'sequential_processing': True,
        'cache_predictions': False
    },
    'data': {
        'dtype': 'float32',
        'chunk_size': 10000
    }
}

# Speed-optimized configuration
speed_config = {
    'name': 'Speed Optimized',
    'description': 'For quick iterations',
    'models': {
        'batch_size': 256,
        'max_steps': 1000,
        'val_check_steps': 50,
        'early_stop_patience_steps': 50
    },
    'cv': {
        'n_windows': 2,
        'parallel_models': True,
        'n_jobs': -1
    },
    'features': {
        'max_features': 50,
        'feature_selection': 'mutual_info'
    }
}

# GPU-optimized configuration
gpu_config = {
    'name': 'GPU Optimized',
    'description': 'For CUDA-enabled systems',
    'models': {
        'batch_size': 512,
        'max_steps': 20000,
        'use_amp': True,  # Mixed precision
        'num_workers': 4  # Data loading
    },
    'cv': {
        'n_windows': 10,
        'gpu_memory_fraction': 0.8
    },
    'optimization': {
        'gradient_checkpointing': True,
        'pin_memory': True
    }
}

# Display configurations
for config in [memory_config, speed_config, gpu_config]:
    print(f"\n{config['name']}:")
    print(f"  {config['description']}")
    print("  Settings:")
    for category, settings in config.items():
        if category not in ['name', 'description']:
            print(f"    {category}:")
            for key, value in settings.items():
                print(f"      {key}: {value}")

## 9. Real-time Monitoring Dashboard

In [ ]:
print("=" * 70)
print("PERFORMANCE MONITORING")
print("=" * 70)

class PerformanceMonitor:
    """Monitor system performance during CV."""
    
    def __init__(self):
        self.history = {
            'time': [],
            'cpu': [],
            'memory': [],
            'gpu': []
        }
    
    def update(self):
        """Collect current metrics."""
        self.history['time'].append(time.time())
        self.history['cpu'].append(psutil.cpu_percent())
        self.history['memory'].append(psutil.virtual_memory().percent)
        
        if torch.cuda.is_available():
            gpu_mem = torch.cuda.memory_allocated() / torch.cuda.get_device_properties(0).total_memory * 100
            self.history['gpu'].append(gpu_mem)
        else:
            self.history['gpu'].append(0)
    
    def get_summary(self):
        """Get performance summary."""
        if not self.history['time']:
            return "No data collected"
        
        summary = {
            'CPU': {
                'avg': np.mean(self.history['cpu']),
                'max': np.max(self.history['cpu'])
            },
            'Memory': {
                'avg': np.mean(self.history['memory']),
                'max': np.max(self.history['memory'])
            },
            'GPU': {
                'avg': np.mean(self.history['gpu']),
                'max': np.max(self.history['gpu'])
            }
        }
        return summary

# Simulate monitoring
monitor = PerformanceMonitor()

print("\nSimulating CV run with monitoring...")
for i in range(5):
    # Simulate work
    time.sleep(0.5)
    _ = np.random.randn(1000, 1000) @ np.random.randn(1000, 1000)  # CPU work
    
    # Update monitor
    monitor.update()
    
    # Print current status
    print(f"  Step {i+1}/5: CPU={psutil.cpu_percent():5.1f}%, "
          f"MEM={psutil.virtual_memory().percent:5.1f}%")

# Show summary
print("\nPerformance Summary:")
summary = monitor.get_summary()
for metric, values in summary.items():
    print(f"  {metric}:")
    print(f"    Average: {values['avg']:5.1f}%")
    print(f"    Maximum: {values['max']:5.1f}%")

## Summary

### Key Optimization Strategies:

#### Memory Optimization:
1. Use float32 instead of float64 (50% savings)
2. Process data in chunks for large datasets
3. Load only necessary columns from parquet
4. Clear intermediate variables with gc.collect()
5. Use sequential processing when memory-constrained

#### Speed Optimization:
1. Reduce max_steps for pilot runs
2. Use parallel model training when possible
3. Implement caching for repeated operations
4. Enable early stopping
5. Use appropriate batch sizes

#### GPU Optimization:
1. Enable mixed precision training (AMP)
2. Find optimal batch size for GPU memory
3. Clear cache between models
4. Set memory fraction limits
5. Use gradient checkpointing for large models

### Quick Wins:
- **Pilot config**: 3-5x faster with minimal accuracy loss
- **Float32**: 50% memory reduction
- **Parallel models**: Up to Nx speedup for N models
- **Early stopping**: 20-50% time savings
- **Caching**: 100x speedup for repeated runs

### When to Use What:

| Constraint | Priority | Key Optimizations |
|------------|----------|-------------------|
| Low Memory | Memory > Speed | Float32, chunks, sequential |
| Time Pressure | Speed > Accuracy | Pilot config, parallel, cache |
| GPU Available | Speed + Memory | Large batches, AMP, parallel |
| CPU Only | Balanced | Small batches, multithread |
| Production | Accuracy > Speed | Full config, validation |

### Next Steps:
1. Profile your specific CV pipeline
2. Identify the bottleneck (memory, CPU, GPU, I/O)
3. Apply targeted optimizations
4. Monitor and iterate
5. Document optimal settings for your use case